# Multi-Footprint MODIS CNN/U-Net

This notebook trains on 24x24 MODIS chips where each chip has multiple OCO-2 SIF footprint masks. The model outputs one SIF map per chip, then each valid footprint mask averages that map into a footprint-level prediction.

## 1. Imports

In [ ]:
from collections import OrderedDict
from pathlib import Path
import hashlib
import math
import random

import altair as alt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader, Dataset

alt.data_transformers.disable_max_rows()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

## 2. Configuration

In [ ]:
CHIP_DIR = Path('data/cnn_modis_chips/multisif_24x24_5to10_active_crop')
METADATA_PATH = CHIP_DIR / 'chip_metadata.csv'
OUTPUT_WORK_DIR = Path('data/cnn_modis_models/multisif_24x24')
OUTPUT_WORK_DIR.mkdir(parents=True, exist_ok=True)
STATS_CACHE_DIR = Path('/kaggle/working/normalization_stats')
STATS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Choose one target. NPZ stores all three targets.
# TARGET_COL = 'Daily_SIF_757nm'
# TARGET_COL = 'Daily_SIF_771nm'
TARGET_COL = 'target_modis_sif'

MIN_VALID_TARGET_FOOTPRINTS = 5
MIN_FAPAR_VALID = 0.90
MIN_EVI_VALID = 0.90
MIN_NDVI_VALID = 0.90
MIN_PAR_VALID = 0.01

SPLIT_MODE = 'random_stratified'  # 'random_stratified' or 'year'
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TRAIN_YEARS = [2019, 2020, 2021, 2022]
VAL_YEARS = [2023]
TEST_YEARS = [2024]

BATCH_SIZE = 256
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
SHARD_CACHE_SIZE = 24
MAX_SAMPLES_FOR_STATS = 5000
USE_STATS_CACHE = True
NORMALIZE_TARGET = True

# With NORMALIZE_TARGET=True, HUBER_BETA is in normalized target units.
HUBER_BETA = 0.5

SIF_BIN_EDGES = [-0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0, 1.25]
SIF_BIN_LABELS = [
    '[-0.5,-0.25)', '[-0.25,0)', '[0,0.25)', '[0.25,0.5)',
    '[0.5,0.75)', '[0.75,1)', '[1,1.25]'
]

RUN_NAME = TARGET_COL.replace('Daily_SIF_', 'sif_').replace('target_', '')
LOSS_TAG = f'huber_beta{str(HUBER_BETA).replace(".", "p")}_bs{BATCH_SIZE}'
MODEL_OUT = OUTPUT_WORK_DIR / f'small_unet_multisif_{RUN_NAME}_{LOSS_TAG}.pt'
PREDICTIONS_OUT = OUTPUT_WORK_DIR / f'test_predictions_multisif_{RUN_NAME}_{LOSS_TAG}.csv'

print('chip dir:', CHIP_DIR)
print('target:', TARGET_COL)
print('model out:', MODEL_OUT)

## 3. Index Shards and Metadata

In [ ]:
def build_shard_index(chip_dir: Path) -> tuple[pd.DataFrame, list[str], list[str], list[str]]:
    shard_files = sorted(chip_dir.glob('chips_*.npz'))
    if not shard_files:
        raise FileNotFoundError(f'No chip shards found in {chip_dir}')

    rows = []
    channel_names = None
    target_names = None
    final_check_names = None
    sample_order = 0

    for shard_id, shard_path in enumerate(shard_files):
        with np.load(shard_path, allow_pickle=False) as z:
            chip_ids = [str(x) for x in z['chip_id']]
            if channel_names is None:
                channel_names = [str(x) for x in z['channel_names']]
                target_names = [str(x) for x in z['target_names']]
                final_check_names = [str(x) for x in z['final_check_names']]

            for local_index, chip_id in enumerate(chip_ids):
                rows.append({
                    'sample_order': sample_order,
                    'shard_id': shard_id,
                    'shard_path': str(shard_path),
                    'local_index': local_index,
                    'chip_id': chip_id,
                })
                sample_order += 1

    shard_index = pd.DataFrame(rows)
    duplicated = shard_index[shard_index.duplicated('chip_id', keep=False)]
    if len(duplicated) > 0:
        raise ValueError('Duplicate chip_id values found across shards. Clear stale shard files and regenerate.')

    return shard_index, channel_names, target_names, final_check_names


metadata = pd.read_csv(METADATA_PATH)
shard_index, channel_names, target_names, final_check_names = build_shard_index(CHIP_DIR)
samples = metadata.merge(shard_index, on='chip_id', how='inner')

TARGET_INDEX = target_names.index(TARGET_COL)
FINAL_CHECK_COL = final_check_names[TARGET_INDEX]

print('metadata rows:', len(metadata))
print('indexed chip rows:', len(shard_index))
print('merged rows:', len(samples))
print('channels:', channel_names)
print('targets:', target_names)
print('selected target index:', TARGET_INDEX)
samples.head()

## 4. Filter Chips

In [ ]:
before = len(samples)
accepted_col = f'{TARGET_COL}_accepted_footprints'
if accepted_col not in samples.columns:
    raise ValueError(f'Missing metadata column: {accepted_col}')

samples = samples[
    (samples[accepted_col] >= MIN_VALID_TARGET_FOOTPRINTS)
    & (samples['n_valid_footprints'] >= MIN_VALID_TARGET_FOOTPRINTS)
    & (samples['fapar_valid_fraction'] >= MIN_FAPAR_VALID)
    & (samples['evi_valid_fraction'] >= MIN_EVI_VALID)
    & (samples['ndvi_valid_fraction'] >= MIN_NDVI_VALID)
    & (samples['par_valid_fraction'] >= MIN_PAR_VALID)
].copy()

target_for_bins = samples['mean_target_modis_sif'].clip(
    lower=SIF_BIN_EDGES[0],
    upper=np.nextafter(SIF_BIN_EDGES[-1], SIF_BIN_EDGES[0]),
)
samples['sif_bin'] = pd.cut(
    target_for_bins,
    bins=SIF_BIN_EDGES,
    labels=SIF_BIN_LABELS,
    right=False,
    include_lowest=True,
)
samples = samples.reset_index(drop=True)

print(f'Kept {len(samples):,} / {before:,} chips after filtering')
display(samples[[
    'n_sif_assigned', 'n_valid_footprints', accepted_col,
    'fapar_valid_fraction', 'evi_valid_fraction', 'ndvi_valid_fraction', 'par_valid_fraction',
    'mean_target_modis_sif'
]].describe())

bin_summary = samples.groupby('sif_bin', observed=False).agg(
    n=('chip_id', 'size'),
    mean_chip_target=('mean_target_modis_sif', 'mean'),
).reset_index()
display(bin_summary)

## 5. Train / Validation / Test Split

In [ ]:
def random_stratified_split(table: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(SEED)
    train_idx, val_idx, test_idx = [], [], []
    group_cols = ['states', 'sif_year', 'composite_doy']

    for _, group in table.groupby(group_cols, sort=False):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        if n < 3:
            train_idx.extend(idx)
            continue

        n_test = max(1, int(round(n * TEST_FRAC)))
        n_val = max(1, int(round(n * VAL_FRAC)))
        if n_test + n_val >= n:
            n_test = 1
            n_val = 1

        test_idx.extend(idx[:n_test])
        val_idx.extend(idx[n_test:n_test + n_val])
        train_idx.extend(idx[n_test + n_val:])

    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


def year_split(table: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    train_idx = table.index[table['sif_year'].isin(TRAIN_YEARS)].to_numpy()
    val_idx = table.index[table['sif_year'].isin(VAL_YEARS)].to_numpy()
    test_idx = table.index[table['sif_year'].isin(TEST_YEARS)].to_numpy()
    return train_idx, val_idx, test_idx


if SPLIT_MODE == 'random_stratified':
    train_idx, val_idx, test_idx = random_stratified_split(samples)
elif SPLIT_MODE == 'year':
    train_idx, val_idx, test_idx = year_split(samples)
else:
    raise ValueError(f'Unknown SPLIT_MODE: {SPLIT_MODE}')

print('train:', len(train_idx))
print('val:', len(val_idx))
print('test:', len(test_idx))
pd.Series({'train': len(train_idx), 'val': len(val_idx), 'test': len(test_idx)}) / len(samples)

## 6. Dataset, Shard Cache, and Normalization

In [ ]:
class ShardCache:
    def __init__(self, max_shards: int = 12):
        self.max_shards = max_shards
        self.cache = OrderedDict()

    def get(self, shard_path: str) -> dict[str, np.ndarray]:
        if shard_path in self.cache:
            self.cache.move_to_end(shard_path)
            return self.cache[shard_path]

        with np.load(shard_path, allow_pickle=False) as z:
            shard = {
                'X': z['X'].astype(np.float32),
                'footprint_masks': z['footprint_masks'].astype(np.float32),
                'y_targets': z['y_targets'].astype(np.float32),
                'target_accept': z['target_accept'].astype(np.uint8),
                'footprint_valid': z['footprint_valid'].astype(np.uint8),
                'sif_row_ids': z['sif_row_ids'].astype(np.int64),
            }

        self.cache[shard_path] = shard
        self.cache.move_to_end(shard_path)
        while len(self.cache) > self.max_shards:
            self.cache.popitem(last=False)
        return shard


class ModisMultiSifDataset(Dataset):
    def __init__(
        self,
        table: pd.DataFrame,
        target_index: int,
        channel_mean: np.ndarray | None = None,
        channel_std: np.ndarray | None = None,
        target_mean: float | None = None,
        target_std: float | None = None,
        cache_size: int = 12,
    ):
        self.table = table.reset_index(drop=True).copy()
        self.target_index = target_index
        self.channel_mean = channel_mean
        self.channel_std = channel_std
        self.target_mean = target_mean
        self.target_std = target_std
        self.cache = ShardCache(max_shards=cache_size)

    def __len__(self):
        return len(self.table)

    def __getitem__(self, idx: int):
        row = self.table.iloc[idx]
        shard = self.cache.get(row['shard_path'])
        local_index = int(row['local_index'])

        x = shard['X'][local_index].copy()
        masks = shard['footprint_masks'][local_index].copy()
        y = shard['y_targets'][local_index, :, self.target_index].copy()
        accept = shard['target_accept'][local_index, :, self.target_index].astype(bool)
        valid = shard['footprint_valid'][local_index].astype(bool) & accept & np.isfinite(y)
        sif_ids = shard['sif_row_ids'][local_index].copy()

        y[~np.isfinite(y)] = 0.0
        if self.channel_mean is not None and self.channel_std is not None:
            x = (x - self.channel_mean[:, None, None]) / self.channel_std[:, None, None]

        if self.target_mean is not None and self.target_std is not None:
            y = (y - self.target_mean) / self.target_std
            y[~valid] = 0.0

        return (
            torch.from_numpy(x.astype(np.float32)),
            torch.from_numpy(masks.astype(np.float32)),
            torch.from_numpy(y.astype(np.float32)),
            torch.from_numpy(valid.astype(np.float32)),
            torch.from_numpy(sif_ids.astype(np.int64)),
        )


def compute_channel_stats(table: pd.DataFrame, max_samples: int = 5000) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(SEED)
    if len(table) > max_samples:
        stat_table = table.iloc[rng.choice(len(table), size=max_samples, replace=False)].copy()
    else:
        stat_table = table.copy()

    ds = ModisMultiSifDataset(stat_table, target_index=TARGET_INDEX, cache_size=SHARD_CACHE_SIZE)
    channel_sum = None
    channel_sumsq = None
    pixel_count = 0

    for i in range(len(ds)):
        x, _, _, _, _ = ds[i]
        arr = x.numpy()
        if channel_sum is None:
            channel_sum = np.zeros(arr.shape[0], dtype=np.float64)
            channel_sumsq = np.zeros(arr.shape[0], dtype=np.float64)
        channel_sum += arr.sum(axis=(1, 2))
        channel_sumsq += (arr ** 2).sum(axis=(1, 2))
        pixel_count += arr.shape[1] * arr.shape[2]

    mean = channel_sum / pixel_count
    var = np.maximum(channel_sumsq / pixel_count - mean ** 2, 1e-8)
    return mean.astype(np.float32), np.sqrt(var).astype(np.float32)


def collect_valid_targets(table: pd.DataFrame) -> np.ndarray:
    ds = ModisMultiSifDataset(table, target_index=TARGET_INDEX, cache_size=SHARD_CACHE_SIZE)
    values = []
    for i in range(len(ds)):
        _, _, y, valid, _ = ds[i]
        arr = y.numpy()
        ok = valid.numpy().astype(bool)
        values.append(arr[ok])
    if not values:
        raise ValueError('No valid target values found')
    return np.concatenate(values)


def make_stats_cache_path(table: pd.DataFrame) -> Path:
    cache_fields = [
        f'target={TARGET_COL}',
        f'target_index={TARGET_INDEX}',
        f'split_mode={SPLIT_MODE}',
        f'seed={SEED}',
        f'max_samples_for_stats={MAX_SAMPLES_FOR_STATS}',
        f'normalize_target={NORMALIZE_TARGET}',
        f'n_train={len(table)}',
        'channels=' + ','.join(channel_names),
    ]
    for shard_path in sorted(table['shard_path'].astype(str).unique()):
        shard_file = Path(shard_path)
        if shard_file.exists():
            stat = shard_file.stat()
            cache_fields.append(f'shard={shard_file.name}:{stat.st_size}:{stat.st_mtime_ns}')
        else:
            cache_fields.append(f'shard={shard_path}')
    cache_text = '\n'.join(cache_fields + table['chip_id'].astype(str).tolist())
    cache_hash = hashlib.md5(cache_text.encode('utf-8')).hexdigest()[:16]
    return STATS_CACHE_DIR / f'normalization_stats_{RUN_NAME}_{cache_hash}.npz'


def load_or_compute_normalization_stats(
    table: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, float | None, float | None, Path, bool]:
    stats_cache_path = make_stats_cache_path(table)

    if USE_STATS_CACHE and stats_cache_path.exists():
        print(f'Loading cached normalization stats: {stats_cache_path}')
        with np.load(stats_cache_path, allow_pickle=False) as z:
            cached_channel_names = [str(x) for x in z['channel_names']]
            if cached_channel_names != channel_names:
                raise ValueError('Cached channel names do not match current chip channels.')

            channel_mean = z['channel_mean'].astype(np.float32)
            channel_std = z['channel_std'].astype(np.float32)
            cached_target_mean = float(z['target_mean'])
            cached_target_std = float(z['target_std'])

        if NORMALIZE_TARGET:
            target_mean = cached_target_mean
            target_std = cached_target_std
        else:
            target_mean = None
            target_std = None

        return channel_mean, channel_std, target_mean, target_std, stats_cache_path, True

    print('Computing normalization stats from training chips...')
    channel_mean, channel_std = compute_channel_stats(table, max_samples=MAX_SAMPLES_FOR_STATS)

    if NORMALIZE_TARGET:
        train_targets = collect_valid_targets(table)
        target_mean = float(np.mean(train_targets))
        target_std = float(np.std(train_targets, ddof=1))
    else:
        target_mean = None
        target_std = None

    if USE_STATS_CACHE:
        np.savez_compressed(
            stats_cache_path,
            channel_mean=channel_mean.astype(np.float32),
            channel_std=channel_std.astype(np.float32),
            target_mean=np.array(target_mean if target_mean is not None else np.nan, dtype=np.float32),
            target_std=np.array(target_std if target_std is not None else np.nan, dtype=np.float32),
            channel_names=np.asarray(channel_names),
            target_col=np.asarray([TARGET_COL]),
            split_mode=np.asarray([SPLIT_MODE]),
            max_samples_for_stats=np.asarray([MAX_SAMPLES_FOR_STATS]),
            seed=np.asarray([SEED]),
            n_train_chips=np.asarray([len(table)]),
        )
        print(f'Saved normalization stats: {stats_cache_path}')

    return channel_mean, channel_std, target_mean, target_std, stats_cache_path, False


train_table = samples.loc[train_idx].reset_index(drop=True)
val_table = samples.loc[val_idx].reset_index(drop=True)
test_table = samples.loc[test_idx].reset_index(drop=True)

# Sort by shard for faster NPZ loading.
train_table = train_table.sort_values(['shard_id', 'local_index']).reset_index(drop=True)
val_table = val_table.sort_values(['shard_id', 'local_index']).reset_index(drop=True)
test_table = test_table.sort_values(['shard_id', 'local_index']).reset_index(drop=True)

channel_mean, channel_std, target_mean, target_std, STATS_CACHE_PATH, loaded_stats_cache = load_or_compute_normalization_stats(train_table)

stats_df = pd.DataFrame({'channel': channel_names, 'mean': channel_mean, 'std': channel_std})
display(stats_df)
print('stats_cache_path:', STATS_CACHE_PATH)
print('loaded_stats_cache:', loaded_stats_cache)
print('target_mean:', target_mean)
print('target_std:', target_std)

## 7. DataLoaders

In [ ]:
train_ds = ModisMultiSifDataset(
    train_table,
    target_index=TARGET_INDEX,
    channel_mean=channel_mean,
    channel_std=channel_std,
    target_mean=target_mean,
    target_std=target_std,
    cache_size=SHARD_CACHE_SIZE,
)
val_ds = ModisMultiSifDataset(
    val_table,
    target_index=TARGET_INDEX,
    channel_mean=channel_mean,
    channel_std=channel_std,
    target_mean=target_mean,
    target_std=target_std,
    cache_size=SHARD_CACHE_SIZE,
)
test_ds = ModisMultiSifDataset(
    test_table,
    target_index=TARGET_INDEX,
    channel_mean=channel_mean,
    channel_std=channel_std,
    target_mean=target_mean,
    target_std=target_std,
    cache_size=SHARD_CACHE_SIZE,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

xb, mb, yb, vb, idsb = next(iter(train_loader))
print('X:', xb.shape)
print('masks:', mb.shape)
print('y:', yb.shape)
print('valid:', vb.shape, 'valid footprint counts:', vb.sum(dim=1)[:10].tolist())
print('sif ids:', idsb.shape)

## 8. Small U-Net

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    def __init__(self, in_channels: int, base_channels: int = 32):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_channels * 2, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.out(d1)


model = SmallUNet(in_channels=len(channel_names), base_channels=32).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
model

## 9. Multi-Footprint Loss and Metrics

In [ ]:
def masked_average_multi(pred_map: torch.Tensor, footprint_masks: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    pred = pred_map[:, 0, :, :][:, None, :, :]
    numerator = (pred * footprint_masks).sum(dim=(2, 3))
    denominator = footprint_masks.sum(dim=(2, 3)).clamp_min(eps)
    return numerator / denominator


def chip_balanced_huber_loss(pred_sif: torch.Tensor, y: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    loss_per_footprint = F.smooth_l1_loss(pred_sif, y, beta=HUBER_BETA, reduction='none')
    valid_count = valid.sum(dim=1).clamp_min(1.0)
    loss_per_chip = (loss_per_footprint * valid).sum(dim=1) / valid_count
    return loss_per_chip.mean()


def invert_target_array(y: np.ndarray) -> np.ndarray:
    if target_mean is None or target_std is None:
        return y
    return y * target_std + target_mean


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    residual = y_pred - y_true
    mse = float(np.mean(residual ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(residual)))
    bias = float(np.mean(residual))
    denom = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = float(1.0 - np.sum(residual ** 2) / denom) if denom > 0 else np.nan
    return {'rmse': rmse, 'mae': mae, 'bias': bias, 'r2': r2}


def fit_linear_calibration(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    if np.nanstd(y_pred) == 0:
        return {'intercept': float(np.nanmean(y_true)), 'slope': 0.0}
    slope, intercept = np.polyfit(y_pred, y_true, deg=1)
    return {'intercept': float(intercept), 'slope': float(slope)}


def apply_linear_calibration(y_pred: np.ndarray, calibration: dict[str, float]) -> np.ndarray:
    return calibration['intercept'] + calibration['slope'] * y_pred


@torch.no_grad()
def predict_table(model: nn.Module, loader: DataLoader, table: pd.DataFrame) -> pd.DataFrame:
    model.eval()
    rows = []
    offset = 0

    for xb, masks, yb, valid, sif_ids in loader:
        xb = xb.to(DEVICE)
        masks_gpu = masks.to(DEVICE)
        pred_map = model(xb)
        pred_sif_norm = masked_average_multi(pred_map, masks_gpu).cpu().numpy()

        y_norm = yb.numpy()
        valid_np = valid.numpy().astype(bool)
        sif_ids_np = sif_ids.numpy()
        batch_size = xb.shape[0]

        for b in range(batch_size):
            chip_row = table.iloc[offset + b]
            valid_slots = np.where(valid_np[b])[0]
            y_true = invert_target_array(y_norm[b, valid_slots])
            y_pred = invert_target_array(pred_sif_norm[b, valid_slots])

            for j, slot in enumerate(valid_slots):
                rows.append({
                    'chip_id': chip_row['chip_id'],
                    'slot': int(slot),
                    'sif_row_id': int(sif_ids_np[b, slot]),
                    'Delta_Date': chip_row['Delta_Date'],
                    'states': chip_row.get('states', ''),
                    'hzs_values': chip_row.get('hzs_values', ''),
                    'sif_year': int(chip_row['sif_year']),
                    'sif_doy': int(chip_row['sif_doy']),
                    'composite_doy': int(chip_row['composite_doy']),
                    'par_doy': int(chip_row['par_doy']),
                    'n_valid_footprints': int(chip_row['n_valid_footprints']),
                    'observed_sif': float(y_true[j]),
                    'predicted_sif_raw': float(y_pred[j]),
                })
        offset += batch_size

    return pd.DataFrame(rows)


def evaluate(model: nn.Module, loader: DataLoader, table: pd.DataFrame) -> dict[str, float]:
    pred_df = predict_table(model, loader, table)
    return regression_metrics(pred_df['observed_sif'].to_numpy(), pred_df['predicted_sif_raw'].to_numpy())

## 10. Training

In [ ]:
history = []
best_val_rmse = math.inf
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for xb, masks, yb, valid, _ in train_loader:
        xb = xb.to(DEVICE)
        masks = masks.to(DEVICE)
        yb = yb.to(DEVICE)
        valid = valid.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred_map = model(xb)
        pred_sif = masked_average_multi(pred_map, masks)
        loss = chip_balanced_huber_loss(pred_sif, yb, valid)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_metrics = evaluate(model, val_loader, val_table)
    train_loss = float(np.mean(train_losses))
    row = {'epoch': epoch, 'train_loss': train_loss, **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)

    print(
        f'Epoch {epoch:03d} | train_loss={train_loss:.5f} | '
        f"val_rmse={val_metrics['rmse']:.5f} | val_r2={val_metrics['r2']:.4f} | "
        f"val_mae={val_metrics['mae']:.5f} | val_bias={val_metrics['bias']:.5f}"
    )

    if val_metrics['rmse'] < best_val_rmse:
        best_val_rmse = val_metrics['rmse']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df.tail()

## 11. Calibration and Test Metrics

In [ ]:
val_pred = predict_table(model, val_loader, val_table)
calibration = fit_linear_calibration(
    val_pred['observed_sif'].to_numpy(),
    val_pred['predicted_sif_raw'].to_numpy(),
)
val_pred['predicted_sif_calibrated'] = apply_linear_calibration(
    val_pred['predicted_sif_raw'].to_numpy(), calibration
)
val_metrics_raw = regression_metrics(val_pred['observed_sif'].to_numpy(), val_pred['predicted_sif_raw'].to_numpy())
val_metrics_calibrated = regression_metrics(val_pred['observed_sif'].to_numpy(), val_pred['predicted_sif_calibrated'].to_numpy())

test_pred = predict_table(model, test_loader, test_table)
test_pred['predicted_sif_calibrated'] = apply_linear_calibration(
    test_pred['predicted_sif_raw'].to_numpy(), calibration
)
test_pred['predicted_sif'] = test_pred['predicted_sif_calibrated']
test_pred['residual_raw'] = test_pred['predicted_sif_raw'] - test_pred['observed_sif']
test_pred['residual_calibrated'] = test_pred['predicted_sif_calibrated'] - test_pred['observed_sif']
test_pred['residual'] = test_pred['residual_calibrated']

test_metrics_raw = regression_metrics(test_pred['observed_sif'].to_numpy(), test_pred['predicted_sif_raw'].to_numpy())
test_metrics_calibrated = regression_metrics(test_pred['observed_sif'].to_numpy(), test_pred['predicted_sif_calibrated'].to_numpy())
test_metrics = test_metrics_calibrated

observed_for_bins = test_pred['observed_sif'].clip(
    lower=SIF_BIN_EDGES[0],
    upper=np.nextafter(SIF_BIN_EDGES[-1], SIF_BIN_EDGES[0]),
)
test_pred['observed_sif_bin'] = pd.cut(
    observed_for_bins,
    bins=SIF_BIN_EDGES,
    labels=SIF_BIN_LABELS,
    right=False,
    include_lowest=True,
)

def prediction_bin_metrics(table: pd.DataFrame, pred_col: str, prefix: str) -> pd.DataFrame:
    rows = []
    for bin_label in SIF_BIN_LABELS:
        group = table[table['observed_sif_bin'].astype(str) == bin_label]
        row = {'sif_bin': bin_label, 'n': int(len(group))}
        if len(group) == 0:
            row.update({
                'observed_min': np.nan,
                'observed_max': np.nan,
                'observed_mean': np.nan,
                f'{prefix}_predicted_mean': np.nan,
                f'{prefix}_rmse': np.nan,
                f'{prefix}_mae': np.nan,
                f'{prefix}_bias': np.nan,
                f'{prefix}_r2': np.nan,
            })
        else:
            y_true_bin = group['observed_sif'].to_numpy()
            y_pred_bin = group[pred_col].to_numpy()
            metrics = regression_metrics(y_true_bin, y_pred_bin)
            row.update({
                'observed_min': float(np.min(y_true_bin)),
                'observed_max': float(np.max(y_true_bin)),
                'observed_mean': float(np.mean(y_true_bin)),
                f'{prefix}_predicted_mean': float(np.mean(y_pred_bin)),
                f'{prefix}_rmse': metrics['rmse'],
                f'{prefix}_mae': metrics['mae'],
                f'{prefix}_bias': metrics['bias'],
                f'{prefix}_r2': metrics['r2'],
            })
        rows.append(row)
    return pd.DataFrame(rows)


raw_bin_metrics = prediction_bin_metrics(test_pred, 'predicted_sif_raw', 'raw')
calibrated_bin_metrics = prediction_bin_metrics(test_pred, 'predicted_sif_calibrated', 'calibrated')
test_bin_metrics = raw_bin_metrics.merge(
    calibrated_bin_metrics.drop(columns=['n', 'observed_min', 'observed_max', 'observed_mean']),
    on='sif_bin',
    how='left',
)

BIN_METRICS_OUT = PREDICTIONS_OUT.with_name(f'{PREDICTIONS_OUT.stem}_bin_metrics.csv')
test_pred.to_csv(PREDICTIONS_OUT, index=False)
test_bin_metrics.to_csv(BIN_METRICS_OUT, index=False)

print('Validation metrics raw:')
print(val_metrics_raw)
print('\nValidation metrics calibrated:')
print(val_metrics_calibrated)
print('\nTest metrics raw:')
print(test_metrics_raw)
print('\nTest metrics calibrated:')
print(test_metrics_calibrated)
print('\nCalibration:')
print(calibration)
print(f'\nSaved test predictions to {PREDICTIONS_OUT}')
print(f'Saved test bin metrics to {BIN_METRICS_OUT}')
display(test_pred.head())
test_bin_metrics

## 12. Diagnostics

In [ ]:
train_loss_chart = (
    alt.Chart(history_df)
    .mark_line(point=True)
    .encode(
        x=alt.X('epoch:Q', title='Epoch'),
        y=alt.Y('train_loss:Q', title='Huber loss'),
        tooltip=['epoch:Q', alt.Tooltip('train_loss:Q', format='.5f')],
    )
    .properties(width=420, height=260, title='Training Loss')
)

val_rmse_chart = (
    alt.Chart(history_df)
    .mark_line(point=True, color='#d62728')
    .encode(
        x=alt.X('epoch:Q', title='Epoch'),
        y=alt.Y('val_rmse:Q', title='Validation RMSE'),
        tooltip=['epoch:Q', alt.Tooltip('val_rmse:Q', format='.5f')],
    )
    .properties(width=420, height=260, title='Validation RMSE')
)

display(alt.hconcat(train_loss_chart, val_rmse_chart))

scatter_df = test_pred[['observed_sif', 'predicted_sif']].copy()
plot_min = float(min(scatter_df['observed_sif'].min(), scatter_df['predicted_sif'].min()))
plot_max = float(max(scatter_df['observed_sif'].max(), scatter_df['predicted_sif'].max()))
tick_step = 0.25
domain_min = math.floor(plot_min / tick_step) * tick_step
domain_max = math.ceil(plot_max / tick_step) * tick_step
SIF_PLOT_DOMAIN = [domain_min, domain_max]
SIF_PLOT_TICKS = np.round(np.arange(domain_min, domain_max + tick_step / 2, tick_step), 2).tolist()

line_df = pd.DataFrame({'observed_sif': SIF_PLOT_DOMAIN, 'predicted_sif': SIF_PLOT_DOMAIN})
axis_common = alt.Axis(values=SIF_PLOT_TICKS, format='.2f', tickCount=len(SIF_PLOT_TICKS))

density_chart = (
    alt.Chart(scatter_df)
    .mark_rect()
    .encode(
        x=alt.X(
            'observed_sif:Q',
            bin=alt.Bin(maxbins=90, extent=SIF_PLOT_DOMAIN),
            scale=alt.Scale(domain=SIF_PLOT_DOMAIN, nice=False),
            axis=axis_common,
            title='Observed OCO-2 SIF',
        ),
        y=alt.Y(
            'predicted_sif:Q',
            bin=alt.Bin(maxbins=90, extent=SIF_PLOT_DOMAIN),
            scale=alt.Scale(domain=SIF_PLOT_DOMAIN, nice=False),
            axis=axis_common,
            title='Calibrated predicted footprint-mean SIF',
        ),
        color=alt.Color('count():Q', scale=alt.Scale(type='log', scheme='turbo'), title='Density'),
        tooltip=[alt.Tooltip('count():Q', title='N')],
    )
)
diagonal = alt.Chart(line_df).mark_line(color='black', strokeWidth=1).encode(
    x=alt.X('observed_sif:Q', scale=alt.Scale(domain=SIF_PLOT_DOMAIN, nice=False)),
    y=alt.Y('predicted_sif:Q', scale=alt.Scale(domain=SIF_PLOT_DOMAIN, nice=False)),
)
display((density_chart + diagonal).properties(
    width=520,
    height=520,
    title=f"Calibrated Test RMSE={test_metrics['rmse']:.4f}, R2={test_metrics['r2']:.3f}",
))

residual_chart = (
    alt.Chart(test_pred)
    .mark_bar(color='#4C78A8')
    .encode(
        x=alt.X('residual_calibrated:Q', bin=alt.Bin(maxbins=80), title='Calibrated predicted - observed SIF'),
        y=alt.Y('count():Q', title='Count'),
        tooltip=[alt.Tooltip('count():Q', title='N')],
    )
    .properties(width=620, height=300, title='Calibrated Residual Distribution')
)
display(residual_chart)

## 13. Visualize One Multi-Footprint Chip

In [ ]:
sample_idx = 0
model.eval()

x, masks, y, valid, sif_ids = test_ds[sample_idx]
with torch.no_grad():
    pred_map_norm = model(x[None].to(DEVICE)).cpu()[0, 0].numpy()

pred_map = invert_target_array(pred_map_norm)
valid_np = valid.numpy().astype(bool)
masks_np = masks.numpy()
mask_union = np.clip(masks_np[valid_np].sum(axis=0), 0, 1)

def chip_to_long(arr: np.ndarray, value_col: str) -> pd.DataFrame:
    yy, xx = np.indices(arr.shape)
    return pd.DataFrame({'x': xx.ravel(), 'y': yy.ravel(), value_col: arr.ravel()})


pred_df = chip_to_long(pred_map, 'pred_sif')
mask_df = chip_to_long(mask_union, 'mask_union')
overlay_df = pred_df.merge(mask_df, on=['x', 'y'])

axis_x = alt.X('x:O', axis=None, title=None)
axis_y = alt.Y('y:O', sort='descending', axis=None, title=None)

pred_chart = alt.Chart(pred_df).mark_rect().encode(
    x=axis_x,
    y=axis_y,
    color=alt.Color('pred_sif:Q', scale=alt.Scale(scheme='viridis'), title='SIF'),
    tooltip=['x:O', 'y:O', alt.Tooltip('pred_sif:Q', format='.4f')],
).properties(width=250, height=250, title='Predicted SIF map')

mask_chart = alt.Chart(mask_df).mark_rect().encode(
    x=axis_x,
    y=axis_y,
    color=alt.Color('mask_union:Q', scale=alt.Scale(scheme='magma', domain=[0, 1]), title='Mask'),
    tooltip=['x:O', 'y:O', alt.Tooltip('mask_union:Q', format='.3f')],
).properties(width=250, height=250, title='Union of valid footprint masks')

overlay_base = alt.Chart(overlay_df).mark_rect().encode(
    x=axis_x,
    y=axis_y,
    color=alt.Color('pred_sif:Q', scale=alt.Scale(scheme='viridis'), title='SIF'),
)
overlay_mask = alt.Chart(overlay_df).mark_rect(color='#d62728').encode(
    x=axis_x,
    y=axis_y,
    opacity=alt.Opacity('mask_union:Q', scale=alt.Scale(domain=[0, 1], range=[0, 0.65]), legend=None),
)
overlay_chart = (overlay_base + overlay_mask).properties(width=250, height=250, title='Masks over prediction')

display(alt.hconcat(pred_chart, mask_chart, overlay_chart))
print('valid SIF row ids:', sif_ids.numpy()[valid_np])
print('observed targets:', invert_target_array(y.numpy()[valid_np]))

## 14. Save Model

In [ ]:
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'channel_names': channel_names,
        'channel_mean': channel_mean,
        'channel_std': channel_std,
        'target_names': target_names,
        'target_col': TARGET_COL,
        'target_index': TARGET_INDEX,
        'target_mean': target_mean,
        'target_std': target_std,
        'stats_cache_path': str(STATS_CACHE_PATH),
        'loaded_stats_cache': loaded_stats_cache,
        'config': {
            'target_col': TARGET_COL,
            'run_name': RUN_NAME,
            'split_mode': SPLIT_MODE,
            'loss_tag': LOSS_TAG,
            'huber_beta': HUBER_BETA,
            'batch_size': BATCH_SIZE,
            'epochs': EPOCHS,
            'learning_rate': LEARNING_RATE,
            'weight_decay': WEIGHT_DECAY,
            'min_valid_target_footprints': MIN_VALID_TARGET_FOOTPRINTS,
            'max_samples_for_stats': MAX_SAMPLES_FOR_STATS,
            'use_stats_cache': USE_STATS_CACHE,
        },
        'calibration': calibration,
        'val_metrics_raw': val_metrics_raw,
        'val_metrics_calibrated': val_metrics_calibrated,
        'test_metrics_raw': test_metrics_raw,
        'test_metrics_calibrated': test_metrics_calibrated,
        'test_bin_metrics': test_bin_metrics.to_dict('records'),
    },
    MODEL_OUT,
)

print(f'Saved model to {MODEL_OUT}')
print(f'Saved test predictions to {PREDICTIONS_OUT}')
print(f'Saved test bin metrics to {BIN_METRICS_OUT}')